<div style="direction: rtl; white-space: normal; line-height: 1;">
# Build Quran Embeddings

در این مرحله:

- دیتاست تمیزشده قرآن خوانده می‌شود.
- متن هر آیه برای embedding آماده می‌شود.
- مدل چندزبانه بارگذاری می‌شود.
- بردارهای آیات ساخته و ذخیره می‌شوند.
</div>

In [1]:
import sys
import platform

print("Python:", sys.version)
print("Machine:", platform.machine())
print("Executable:", sys.executable)

Python: 3.11.3 (v3.11.3:f3909b8bc8, Apr  4 2023, 20:12:10) [Clang 13.0.0 (clang-1300.0.29.30)]
Machine: x86_64
Executable: /usr/local/bin/python3


In [2]:
%pip install --force-reinstall "numpy==1.24.3"

  Using cached numpy-1.24.3-cp311-cp311-macosx_10_9_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.24.3-cp311-cp311-macosx_10_9_x86_64.whl (19.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.3
    Uninstalling numpy-1.24.3:
      Successfully uninstalled numpy-1.24.3

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Verify the environment after kernel restart

import numpy as np
import torch
import sentence_transformers

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("MPS available:", torch.backends.mps.is_available())

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NumPy: 1.24.3
PyTorch: 2.2.2
Sentence Transformers: 5.0.0
MPS available: False


<div style="direction: rtl; white-space: normal; line-height: 1;">
فقط مدل را بارگذاری و با یک متن کوتاه تست می‌کنیم.
</div>

In [4]:
# Load embedding model

from sentence_transformers import SentenceTransformer

model_name = "intfloat/multilingual-e5-small"

model = SentenceTransformer(
    model_name,
    device="cpu"
)

test_text = ["passage: خداوند بخشنده و مهربان است"]

test_embedding = model.encode(
    test_text,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("Model:", model_name)
print("Embedding shape:", test_embedding.shape)
print("Embedding dtype:", test_embedding.dtype)

Model: intfloat/multilingual-e5-small
Embedding shape: (1, 384)
Embedding dtype: float32


<div style="direction: rtl; white-space: normal; line-height: 1;">
در این بخش، دیتاست تمیزشده بارگذاری می‌شود و برای هر آیه یک متن بازیابی شامل دو ترجمه فارسی ساخته می‌شود تا پرسش‌های فارسی با دقت بیشتری بازیابی شوند.
</div>

In [5]:
# Load cleaned Quran dataset and prepare Persian passages

from pathlib import Path
import json

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

input_file = project_root / "data" / "processed" / "quran_dataset_clean.json"

with open(input_file, "r", encoding="utf-8") as f:
    quran_data = json.load(f)

passages = []

for item in quran_data:
    passage = (
        f"passage: "
        f"{item['fooladvand_clean']} "
        f"{item['ansarian_clean']}"
    )

    passages.append(passage)

print("Records:", len(quran_data))
print("Passages:", len(passages))
print("\nSample passage:")
print(passages[0])

Records: 6236
Passages: 6236

Sample passage:
passage: به نام خداوند رحمتگر مهربان به نام خدا که رحمتش بی‌اندازه است و مهربانی‌اش همیشگی.


<div style="direction: rtl; white-space: normal; line-height: 1;">
ساخت embedding برای هر ۶۲۳۶ آیه و ذخیره روی دیسک
</div>

In [6]:
# Generate and save Quran embeddings

import numpy as np
from pathlib import Path

embeddings_dir = project_root / "data" / "embeddings"
embeddings_dir.mkdir(parents=True, exist_ok=True)

embeddings_file = embeddings_dir / "quran_embeddings.npy"

quran_embeddings = model.encode(
    passages,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

np.save(embeddings_file, quran_embeddings)

print("Embeddings shape:", quran_embeddings.shape)
print("Embeddings dtype:", quran_embeddings.dtype)
print("Saved:", embeddings_file)

Batches: 100%|██████████| 390/390 [03:39<00:00,  1.78it/s]


Embeddings shape: (6236, 384)
Embeddings dtype: float32
Saved: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG/data/embeddings/quran_embeddings.npy


<div style="direction: rtl; white-space: normal; line-height: 1;">
یک تست Retrieval بگیریم تا مطمئن شویم بردارها واقعاً آیات مرتبط را برمی‌گردانند.
</div>

In [7]:
# Test semantic retrieval

query = "خداوند مهربان و بخشنده است"

query_embedding = model.encode(
    [f"query: {query}"],
    normalize_embeddings=True,
    convert_to_numpy=True
)

scores = quran_embeddings @ query_embedding[0]

top_k = 5
top_indices = np.argsort(scores)[::-1][:top_k]

print("Query:", query)
print()

for rank, index in enumerate(top_indices, start=1):
    item = quran_data[index]

    print(f"Rank {rank}")
    print(f"Score: {scores[index]:.4f}")
    print(f"Surah: {item['surah']} | Ayah: {item['ayah']}")
    print("Arabic:", item["arabic"])
    print("Fooladvand:", item["fooladvand"])
    print("Ansarian:", item["ansarian"])
    print("-" * 80)

Query: خداوند مهربان و بخشنده است

Rank 1
Score: 0.9025
Surah: 1 | Ayah: 3
Arabic: ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
Fooladvand: رحمتگر مهربان،
Ansarian: رحمتش بی اندازه و مهربانی اش همیشگی است.
--------------------------------------------------------------------------------
Rank 2
Score: 0.8998
Surah: 1 | Ayah: 1
Arabic: بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ
Fooladvand: به نام خداوند رحمتگر مهربان
Ansarian: به نام خدا که رحمتش بی‌اندازه است و مهربانی‌اش همیشگی.
--------------------------------------------------------------------------------
Rank 3
Score: 0.8974
Surah: 4 | Ayah: 106
Arabic: وَٱسْتَغْفِرِ ٱللَّهَ إِنَّ ٱللَّهَ كَانَ غَفُورًا رَّحِيمًا
Fooladvand: و از خدا آمرزش بخواه، که خدا آمرزنده مهربان است.
Ansarian: و از خدا آمرزش بخواه؛ زیرا خدا همواره بسیار آمرزنده و مهربان است.
--------------------------------------------------------------------------------
Rank 4
Score: 0.8969
Surah: 78 | Ayah: 37
Arabic: رَّبِّ ٱلسَّمَٰوَٰتِ وَٱلْأَرْضِ وَمَا بَيْنَهُمَا ٱلرَّحْمَٰنِ لَا يَمْلِكُونَ مِ

<div style="direction: rtl; white-space: normal; line-height: 1;">
تست کنیم embedding ذخیره‌شده بعداً بدون ساخت دوباره قابل بارگذاری است.
</div>

In [8]:
# Reload and validate saved embeddings

loaded_embeddings = np.load(embeddings_file)

print("Loaded shape:", loaded_embeddings.shape)
print("Loaded dtype:", loaded_embeddings.dtype)
print(
    "Same values:",
    np.allclose(quran_embeddings, loaded_embeddings)
)

Loaded shape: (6236, 384)
Loaded dtype: float32
Same values: True


<div style="direction: rtl; white-space: normal; line-height: 1;">
Notebook 04 کامل شد
</div>